In [0]:
import yaml
from pyspark.sql import functions as F

config_path = "../conf/base_config.yaml"

try:
    with open(config_path, "r") as file:
        config = yaml.safe_load(file)
except FileNotFoundError:
    raise Exception(f"Arquivo de configuração não encontrado no caminho: {config_path}")

paysim_silver = config['paths']['silver']
paysim_gold = config['paths']['gold']
fraud_types = config['business_rules']['fraud_types']

print(f"Origem (Silver): {paysim_silver}")
print(f"Destino (Gold): {paysim_gold}")
print(f"Tipos de transação analisados: {fraud_types}")

In [0]:
print("Lendo a camada Silver e aplicando filtro de negócio...")
df_silver = spark.read.table(paysim_silver)

# Filtrando os dados dinamicamente

df_filtered = df_silver.filter(F.col("type").isin(fraud_types))

In [0]:
print("Construindo as Features...")

# Lista de tipos onde a conta de Origem RECEBE dinheiro
input_type = ["CASH_IN"] 

df_features = df_filtered \
    .withColumn("error_orig", 
        F.when(F.col("type").isin(input_type), 
             F.col("old_balance_org") + F.col("amount") - F.col("new_balance_orig")
        ).otherwise(
             F.col("old_balance_org") - F.col("amount") - F.col("new_balance_orig")
        )
    ) \
    .withColumn("error_dest", 
        F.when(F.col("name_dest").startswith("M"), 
             F.lit(0.0)
        ).when(F.col("type").isin(input_type), 
             F.col("old_balance_dest") - F.col("amount") - F.col("new_balance_dest")
        ).otherwise(
             F.col("old_balance_dest") + F.col("amount") - F.col("new_balance_dest")
        )
    ) \
    .withColumn("hour_of_day", F.col("step") % 24) \
    .withColumn("ratio_amount_balance", 
        F.when(F.col("old_balance_org") > 0, 
               F.col("amount") / F.col("old_balance_org")
        ).otherwise(-1.0)
    ) \
    .withColumn("type_idx", 
                F.when(F.col("type") == "TRANSFER", 0)
                 .otherwise(1)
    ) \
    .withColumn("dest_is_empty", 
                F.when(F.col("old_balance_dest") == 0, 1)
                 .otherwise(0)
    )

In [0]:
# Seleção das colunas relevantes
features_list = [
    "step", "hour_of_day", "type_idx", "amount",
    "old_balance_org", "error_orig", "old_balance_dest",
    "error_dest", "dest_is_empty", "ratio_amount_balance", "is_fraud"
]

df_gold = df_features.select(features_list)

# Data Quality Check
nulls_target = df_gold.filter(F.col("is_fraud").isNull()).count()
assert nulls_target == 0, f"🛑 Alerta de DQ: {nulls_target} nulos encontrados na variável alvo (is_fraud)!"

print("✅ Qualidade de dados validada: Nenhum valor nulo na variável alvo.")

In [0]:
print(f"Salvando dados na tabela Gold ({paysim_gold})...")

df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(paysim_gold)

print("Camada Silver -> Gold processada com sucesso!")